## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🔄 Azure AI Agent with Existing Agent - Persistent Financial Advisor 🏦

This notebook demonstrates working with pre-existing Azure AI Agents using `FoundryAgent` for production scenarios where you want to reuse a persistent Advisor agent.

## Features Covered:
- Connecting to pre-configured Azure AI Foundry agents by name
- Working with existing agents using `FoundryAgent`
- Adding function tools to existing agents
- Production patterns for reusing Financial Advisor agents

### ⚠️ Important Note ⚠️
> **Persistent agents are useful for production scenarios where you want consistent behavior across sessions. The agent retains its configuration and can be accessed by name.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

This example demonstrates how to work with an existing Azure AI agent by connecting to it using its agent ID.

## Import Libraries

Import the required libraries using `FoundryAgent`:

In [11]:
import os
from pathlib import Path
from random import randint, uniform
from typing import Annotated, Any, Mapping, Sequence

from agent_framework._tools import FunctionInvocationLayer
from agent_framework._middleware import ChatMiddlewareLayer
from agent_framework._types import Message
from agent_framework.foundry import FoundryAgent, RawFoundryAgentChatClient
from agent_framework.observability import ChatTelemetryLayer
from azure.identity import AzureCliCredential
from pydantic import Field


class FoundryAgentChatClient(
    FunctionInvocationLayer,
    ChatMiddlewareLayer,
    ChatTelemetryLayer,
    RawFoundryAgentChatClient,
):
    """Workaround for rc6: strip tool schemas from API request."""

    async def _prepare_options(
        self,
        messages: Sequence[Message],
        options: Mapping[str, Any],
        **kwargs: Any,
    ) -> dict[str, Any]:
        run_options = await super()._prepare_options(messages, options, **kwargs)
        run_options.pop("tools", None)
        run_options.pop("tool_choice", None)
        run_options.pop("parallel_tool_calls", None)
        return run_options

## Initial Setup

Load environment variables from the `.env` file:

In [12]:
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
notebook_path = Path().absolute()
load_dotenv('../../.env')

print("📝 Note: This notebook demonstrates working with existing (persistent) agents")
print("✅ Environment variables and async imports are ready for use")

📝 Note: This notebook demonstrates working with existing (persistent) agents
✅ Environment variables and async imports are ready for use


## Check Environment Variables

Verify that the required environment variables are set:

In [13]:
# Check required environment variables
required_vars = ["AI_FOUNDRY_PROJECT_ENDPOINT", "AZURE_AI_MODEL_DEPLOYMENT_NAME"]
missing_vars = []

for var in required_vars:
    value = os.getenv(var)
    if value:
        print(f"✅ {var}: {value[:50]}..." if len(str(value)) > 50 else f"✅ {var}: {value}")
    else:
        print(f"❌ {var}: Not set")
        missing_vars.append(var)

if missing_vars:
    print(f"\n⚠️  Please set the following environment variables: {', '.join(missing_vars)}")
else:
    print("\n✅ All required environment variables are set!")

✅ AI_FOUNDRY_PROJECT_ENDPOINT: https://demopocaifoundry.services.ai.azure.com/api...
✅ AZURE_AI_MODEL_DEPLOYMENT_NAME: gpt-4o

✅ All required environment variables are set!


## Define Function Tools 🏦

Let's define banking functions that our Financial Advisor agent can use:

In [14]:
def get_account_balance(
    account_id: Annotated[str, Field(description="The customer account ID to check balance for.")],
) -> str:
    """Get the current balance for a customer account."""
    balance = round(uniform(5000, 75000), 2)
    return f"Account {account_id}: Current balance is ${balance:,.2f}"


def get_loan_rates(
    loan_type: Annotated[str, Field(description="Type of loan: mortgage, auto, personal, or business")],
) -> str:
    """Get current interest rates for different loan types."""
    rates = {
        "mortgage": {"rate": 6.5, "term": "30 years"},
        "auto": {"rate": 7.2, "term": "5 years"},
        "personal": {"rate": 10.5, "term": "3 years"},
        "business": {"rate": 8.0, "term": "10 years"}
    }
    loan_type = loan_type.lower()
    if loan_type in rates:
        return f"Current {loan_type} loan rate: {rates[loan_type]['rate']}% APR, typical term: {rates[loan_type]['term']}"
    return f"Unknown loan type. Available: mortgage, auto, personal, business"

## Connect to and Use Existing Agent 🔄

This example shows how to:
1. Connect to a pre-configured agent in Azure AI Foundry using `FoundryAgent`
2. Add custom function tools for banking operations
3. Query the agent with banking-related questions

**Key Pattern**: `FoundryAgent(agent_name=..., project_endpoint=..., credential=...)` connects to an existing agent by name.

In [15]:
async def main():
    """Demonstrate using FoundryAgent to work with an existing Foundry agent.
    
    This function shows the workflow:
    1. Connect to a pre-configured agent in Azure AI Foundry by name
    2. Add custom function tools for banking operations
    3. Use the agent for banking queries
    """
    print("=== 🔄 Foundry Agent with Existing Agent ===")
    
    # Connect to an existing pre-configured agent in Azure AI Foundry
    # The agent_name must match an agent configured in your Foundry project
    agent = FoundryAgent(
        agent_name="financial-services-advisor",
        project_endpoint=os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"],
        credential=AzureCliCredential(),
        instructions="You are a helpful Financial Advisor for a retail bank. Help customers with account inquiries and loan information.",
        tools=[get_account_balance, get_loan_rates],
        client_type=FoundryAgentChatClient,
    )
    
    print(f"✅ Connected to existing Foundry agent: {agent.name}")
    
    # Use the agent for banking queries
    query = "What's the balance for account ACC-PROD-001 and what are the current mortgage rates?"
    print(f"\n🤔 Customer: {query}")
    result = await agent.run(query)
    print(f"🏦 Advisor: {result.text}")

## Execute the Example 🚀

Run the main function to see the existing agent workflow:

In [16]:
# Run the main function
await main()

=== 🔄 Foundry Agent with Existing Agent ===
✅ Connected to existing Foundry agent: None

🤔 Customer: What's the balance for account ACC-PROD-001 and what are the current mortgage rates?
🏦 Advisor: I'm sorry, but I can't access personal account information or retrieve specific details about your account. For information about your account balance, I recommend logging into your online banking account, using your bank's mobile app, or contacting customer service directly. You can also visit a nearby branch and speak with a banker there.

As for current **mortgage rates**, they can vary depending on numerous factors such as your credit score, the type of loan you’re interested in (e.g., fixed-rate or adjustable-rate), the loan term, and market conditions. Mortgage rates can change daily, so it's best to check with your bank or lender for the most accurate and current information. Most banks also have mortgage rate pages on their websites.

If you'd like, I can explain common terms like **A

## Key Takeaways 📚

### Key Methods for Existing Agents

```python
# Connect to an existing pre-configured agent in Azure AI Foundry
agent = FoundryAgent(
    agent_name="FinancialAdvisorAgent",
    project_endpoint=endpoint,
    credential=AzureCliCredential(),
    instructions="...",
    tools=[function1, function2],
    client_type=FoundryAgentChatClient,
)
result = await agent.run("query")
```

### Production Patterns

1. **Pre-configured Agents**: Connect to agents set up in Azure AI Foundry portal
2. **Tool Injection**: Add function tools when creating the FoundryAgent instance
3. **Session Continuity**: Maintain context across customer interactions
4. **Credential Management**: Use sync `AzureCliCredential()` for simplicity

### Use Cases

- **Persistent Financial Advisors**: Same agent persona for all customers
- **Branch-Specific Agents**: Different agents for different bank branches
- **Specialized Advisors**: Mortgage, Investment, Personal Banking agents
- **Session Continuity**: Maintain context across customer interactions

### Best Practices

1. **Agent Naming**: Use clear, descriptive names matching Foundry configuration
2. **Credential Usage**: Use sync `AzureCliCredential()` passed directly to agents
3. **Error Handling**: Handle cases where agent doesn't exist in Foundry
4. **Tool Management**: Add function tools via `tools` parameter
5. **Instructions Override**: Customize agent behavior per session with `instructions`